# Lunar meteoroid impact EMP: first-look SNR at Millstone MISA (440 MHz)

Passive **listen** for a microsecond RF pulse from a hypervelocity impact plasma on the Moon. This is **not** incoherent scatter: there is no long integration. Detection is peak flux versus \(T_\mathrm{sys}\) in a wide band.

**Reference point** (Kesaraju et al. 2016, *Earth Moon Planets*): 1 g at 20 km/s, \(E_k=2\times10^5\) J, RF efficiency \(\eta_\mathrm{RF}=0.01\%\) into \(2\pi\) sr, \(\tau=1~\mu\mathrm{s}\), source bandwidth 10 GHz, receiver 5 MHz. They quoted **86 dB (Arecibo) / 54 dB (Haystack)**. Those numbers are \(20\log_{10}(P_r/kTB)\) (twice a power SNR in dB). The same assumptions give **~44 dB / ~27 dB** as power SNR \(10\log_{10}(P_r/kTB)\). A 2014 Arecibo+Haystack search did **not** confirm EMPs (RFI / Moon-bounce dominated), so \(0.01\%\) is almost certainly optimistic — treat \(\eta_\mathrm{RF}\) as the unknown.

Empty UHF sky is ~35–40 K. Pointed at the Moon, use \(T_\mathrm{moon}\sim200\) K in the beam filling factor, as they did.


## Setup

Received pulse power from an isotropic-into-hemisphere emitter:

\[
S = \frac{\eta_\mathrm{RF} E_k}{\tau}\,\frac{1}{2\pi R^2}\,\frac{B_\mathrm{rx}}{B_\mathrm{src}}
\qquad
P_r = S A_e
\qquad
\mathrm{SNR}_\mathrm{power} = \frac{P_r}{k T_\mathrm{sys} B_\mathrm{rx}}
\]

with \(E_k=\tfrac12 m v^2\) and \(A_e=\eta_A \pi (D/2)^2\).


In [ ]:
from __future__ import annotations

from pathlib import Path

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

Path("figures").mkdir(exist_ok=True)

k_B = 1.380649e-23
c = 2.99792458e8
R_MOON = 384_400e3          # m, Kesaraju et al.
F_UHF = 440e6               # Hz
LAMBDA = c / F_UHF
TAU = 1e-6                  # s
B_SRC = 10e9                # Hz, assumed EMP spectrum
B_RX = 5e6                  # Hz
T_SYS_KES = 250.0           # K: 50 K receiver + 200 K lunar disk (their SNR paragraph)
ETA_ANT = 0.50

def kinetic_energy(mass_kg: np.ndarray, speed_m_s: float) -> np.ndarray:
    return 0.5 * np.asarray(mass_kg, dtype=float) * speed_m_s**2

def collecting_area(diameter_m: float, eta: float = ETA_ANT) -> float:
    return eta * np.pi * (diameter_m / 2.0) ** 2

def flux_in_band(mass_kg, speed_m_s, eta_rf, tau=TAU, b_src=B_SRC, b_rx=B_RX, r=R_MOON):
    p_rf = eta_rf * kinetic_energy(mass_kg, speed_m_s) / tau
    p_band = p_rf * (b_rx / b_src)
    return p_band / (2.0 * np.pi * r**2)   # W / m^2 in B_rx

def received_power(mass_kg, speed_m_s, eta_rf, area_m2, **kw):
    return flux_in_band(mass_kg, speed_m_s, eta_rf, **kw) * area_m2

def snr_power(mass_kg, speed_m_s, eta_rf, area_m2, t_sys=T_SYS_KES, b_rx=B_RX, **kw):
    pr = received_power(mass_kg, speed_m_s, eta_rf, area_m2, b_rx=b_rx, **kw)
    pn = k_B * t_sys * b_rx
    return pr / pn

A_AO = collecting_area(305.0)
A_HO = collecting_area(46.0)
print(f"lambda = {LAMBDA:.3f} m   A_e Arecibo = {A_AO:.0f} m²   A_e MISA = {A_HO:.0f} m²")


## Reproduce the Kesaraju 1 g / 20 km/s / 0.01% point


In [ ]:
m_kes = 1e-3      # kg  (1 g)
v_kes = 20e3      # m/s
eta_kes = 1e-4    # 0.01%

Ek = kinetic_energy(m_kes, v_kes)
S = flux_in_band(m_kes, v_kes, eta_kes)
print(f"E_k = {Ek:.3e} J   (expect 2.00e5)")
print(f"In-band flux = {S:.2e} W/m²   (paper ~1e-14)")

for name, area in [("Arecibo 305 m", A_AO), ("Haystack MISA 46 m", A_HO)]:
    pr = received_power(m_kes, v_kes, eta_kes, area)
    pn = k_B * T_SYS_KES * B_RX
    snr = pr / pn
    print(
        f"{name:22s}  P_r={pr:.2e} W  P_n={pn:.2e} W  "
        f"power SNR={10*np.log10(snr):5.1f} dB  "
        f"20 log10(P_r/P_n)={20*np.log10(snr):5.1f} dB"
    )
print("Paper quoted 86 dB (AO) and 54 dB (HO) = 20 log10 of the power ratio.")


## Millstone MISA: SNR vs RF efficiency and mass

Horizontal dotted line is **power SNR = 10** (10 dB), a rough single-station detection threshold before RFI. The star is the Kesaraju point. If the 2014 search would have seen 27 dB power SNR, a non-detection (or RFI-limited search) implies \(\eta_\mathrm{RF}\) well below \(10^{-4}\) unless the energy is much more broadband than assumed, or beamed away from Earth.

A more realistic MISA \(T_\mathrm{sys}\) on the Moon is closer to \(T_\mathrm{rx}+T_\mathrm{ant}\sim 300{-}400\) K (2014 receive chain ~300 K, partial lunar filling). That shifts SNR down by ~1–2 dB versus the 250 K reference — it does not change the order of magnitude.


In [ ]:
plt.rcParams.update({
    "figure.dpi": 120,
    "font.size": 11,
    "axes.grid": True,
    "grid.alpha": 0.35,
})

eta = np.logspace(-12, -3, 200)
masses_g = np.array([0.01, 0.1, 1.0, 10.0])
v20, v60 = 20e3, 60e3

fig, axes = plt.subplots(1, 2, figsize=(11.2, 4.6), sharey=True)

for m_g in masses_g:
    snr = snr_power(m_g * 1e-3, v20, eta, A_HO)
    axes[0].loglog(eta, snr, label=f"{m_g:g} g")

snr_star = snr_power(m_kes, v_kes, eta_kes, A_HO)
axes[0].scatter([eta_kes], [snr_star], marker="*", s=160, zorder=5, color="k",
                label="Kesaraju 1 g, 20 km/s, 0.01%")
axes[0].axhline(10, color="0.3", ls=":", lw=1.2, label="power SNR = 10")
axes[0].set_xlabel(r"RF efficiency eta_RF = E_RF / E_k")
axes[0].set_ylabel(r"power SNR  P_r / (k T_sys B)")
axes[0].set_title("MISA 440 MHz, 20 km/s, T_sys = 250 K")
axes[0].legend(frameon=False, fontsize=9)

mass_kg = np.logspace(-5, -1, 200)   # 0.01 g → 100 g
for eta_rf, ls in [(1e-4, "-"), (1e-6, "--"), (1e-8, ":")]:
    axes[1].loglog(mass_kg * 1e3, snr_power(mass_kg, v20, eta_rf, A_HO), ls=ls,
                   color="C0", label=f"20 km/s, eta={eta_rf:.0e}")
    axes[1].loglog(mass_kg * 1e3, snr_power(mass_kg, v60, eta_rf, A_HO), ls=ls,
                   color="C1", label=f"60 km/s, eta={eta_rf:.0e}")
axes[1].axhline(10, color="0.3", ls=":", lw=1.2)
axes[1].scatter([1.0], [snr_star], marker="*", s=160, zorder=5, color="k")
axes[1].set_xlabel("meteoroid mass (g)")
axes[1].set_title("MISA · shower-speed vs sporadic")
axes[1].legend(frameon=False, fontsize=8, ncol=1)

for ax in axes:
    ax.set_ylim(1e-4, 1e6)
fig.suptitle("Passive lunar-impact EMP detectability, Haystack MISA (46 m, 50% aperture)", y=1.02)
fig.tight_layout()
fig.savefig("figures/lunar_impact_misa_snr.png", bbox_inches="tight")
plt.show()


## \(\eta_\mathrm{RF}\) needed for power SNR = 10 at MISA


In [ ]:
def eta_for_snr(snr_target, mass_kg, speed_m_s, area_m2, t_sys=T_SYS_KES):
    # SNR scales linearly with eta_rf
    snr1 = snr_power(mass_kg, speed_m_s, 1.0, area_m2, t_sys=t_sys)
    return snr_target / snr1

print("eta_RF for power SNR=10 at MISA (T_sys=250 K, B=5 MHz, tau=1 us, B_src=10 GHz)")
print(f"{'mass':>8}  {'20 km/s':>10}  {'60 km/s':>10}")
for m_g in [0.01, 0.1, 1.0, 10.0, 100.0]:
    e20 = eta_for_snr(10, m_g * 1e-3, v20, A_HO)
    e60 = eta_for_snr(10, m_g * 1e-3, v60, A_HO)
    print(f"{m_g:7g} g  {e20:10.2e}  {e60:10.2e}")

print()
print("If 2014 would have detected the 27 dB Kesaraju case, a clean non-detection")
print("would bound eta_RF << 1e-4 for gram-class 20 km/s impacts (or B_src >> 10 GHz).")
print("Next: replace eta_rf * (B_rx/B_src) with a Close-style chirp occupying UHF,")
print("and require optical-flash + two-station delay coincidence against UHF RFI.")


## How to read this

- **Above the dotted line** is plausibly detectable in a *quiet* 5 MHz channel. Millstone UHF is not quiet; Kesaraju et al. needed Arecibo coincidence and still had Moon-bounce radars.
- **Star** = published optimistic point. Lab RF from hypervelocity impacts is real, but often needs \(v\gtrsim15{-}20\) km/s and (in the Stanford spacecraft work) surface charging. Lunar regolith is not a biased spacecraft.
- SNR scales as \(m\,v^2\,\eta_\mathrm{RF}\,A_e/(T_\mathrm{sys} B_\mathrm{src})\). Doubling bandwidth without capturing more pulse energy **hurts**.
- Empty-sky 35–40 K only applies **off** the Moon. On-Moon, start from ~200 K disk brightness as in the paper.

Papers: Kesaraju et al. 2016, 2019; Close/Nuttall/Johnson Stanford HVI-RF; Suggs et al. optical lunar flashes.
